# 01 — Entrenamiento Random Forest
**Target:** `precio_total_usd`  
**Features:** zona, anillo, superficie, habitaciones, baños, antigüedad, extras


In [ ]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

CSV_PATH   = '../ml_models/datasets/ML_Supervisado_Prediccion_Precio.csv'
MODEL_PATH = '../ml_models/trained/random_forest_v1.pkl'
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head(3)


In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────
# Encoders para categóricas
le_zona   = LabelEncoder()
le_estado = LabelEncoder()
le_mat    = LabelEncoder()
le_tipo   = LabelEncoder()

df['zona_enc']   = le_zona.fit_transform(df['zona'])
df['estado_enc'] = le_estado.fit_transform(df['estado_conservacion'])
df['mat_enc']    = le_mat.fit_transform(df['material_construccion'])
df['tipo_enc']   = le_tipo.fit_transform(df['tipo_propiedad'])

FEATURES = [
    'zona_enc','anillo_vial','tipo_enc','superficie_total_m2',
    'superficie_construida_m2','habitaciones','banos','pisos',
    'garage_vehiculos','antiguedad_anos','estado_enc','mat_enc',
    'tiene_piscina','tiene_jardin','amueblado','tiene_ascensor','seguridad_24h',
]
TARGET = 'precio_total_usd'

X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')


In [ ]:
# ── Entrenamiento con MLflow ──────────────────────────────────────────────
mlflow.set_experiment('random_forest_precio')

with mlflow.start_run(run_name='rf_v1'):
    params = dict(n_estimators=200, max_depth=20, min_samples_leaf=5,
                  n_jobs=-1, random_state=42)
    mlflow.log_params(params)

    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    mlflow.log_metric('MAE', mae)
    mlflow.log_metric('R2',  r2)
    mlflow.sklearn.log_model(model, 'model')

    print(f'MAE: ${mae:,.0f} | R²: {r2:.4f}')


In [ ]:
# ── Guardar modelo ────────────────────────────────────────────────────────
Path('../ml_models/trained').mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
print(f'Modelo guardado en: {MODEL_PATH}')

# Feature importance
fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(fi.head(10))


In [ ]:
# ── Test de inferencia (simula lo que hace random_forest_service.py) ─────
sample = {
    'metros_cuad': 150, 'habitaciones': 3, 'banos': 2,
    'anio_construc': 2018, 'barrio': 'Equipetrol Norte', 'ciudad': 'Santa Cruz'
}
preds = np.array([t.predict([[100,3,3,2,2,3,2,1,1,5,1,1,0,1,0,0,1]])[0] for t in model.estimators_])
print(f'Estimado: ${np.median(preds):,.0f}')
print(f'Rango:    ${np.percentile(preds,5):,.0f} – ${np.percentile(preds,95):,.0f}')
